# azure-foundry — capture works; the money does not, until you say so

Microsoft Foundry models go through the standard `openai` SDK, so capture is free. Then you look at the cost and it is **`None`** — you called your *deployment name*, not a model id.

> **Offline.** No API key, no network — the provider is a fake with the real client's *shape*, or a
> committed cassette. This notebook runs in CI on Python 3.11 and 3.13 via `nbmake`, so if a cell
> below stops working the build goes red.
>
> Beside it, [`main.py`](main.py) is the same story as a script. The last cell here asserts what
> that script asserts.

In [ ]:
# The notebook sits beside the recipe, so its own module is importable. Everything below reuses the
# recipe's fixtures rather than re-inventing them — a notebook that built its own fake could drift
# away from what `main.py` proves and nobody would notice.
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd()))

## The five steps

Every recipe in `providers/` walks the same five, in the same order:

| # | Step | Here |
|---|---|---|
| 1 | **connect** | the standard `openai` SDK at `<endpoint>/openai/v1/` — no `AzureOpenAI`, no `api-version` |
| 2 | **instrument** | one wrap — detection is structural, not name-based |
| 3 | **govern** | a `tokenguard` budget **and** a `guardrails` gate |
| 4 | **record** | `cassette` — the same call replayed offline, 0 provider calls |
| 5 | **prove** | `acttrace` `verify()` and a cost that came from `prices` |

**Distinctive here: money, and only money.** A USD cap on an unpriced deployment counts every call as zero and never binds — the failure that costs real money, because the cap is in your code, your review and your runbook, and it is enforcing nothing.

## 1–2 · Connect and instrument

In [ ]:
import main as recipe
from cendor.core import bus, instrument, prices
from cendor.core.types import LLMCall

seen = []
bus.subscribe(lambda e: seen.append(e) if isinstance(e, LLMCall) else None)
client = instrument(recipe.fake_openai_client())
print(f"deployment: {recipe.DEPLOYMENT}   base model: {recipe.BASE_MODEL}")

## 3 · As shipped, the cap does nothing

In [ ]:
import warnings

from cendor.tokenguard import BudgetExceeded, budget, reset

reset()
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    try:
        with budget(usd=0.000_01, on_exceed="block"):
            recipe.ask(client)
    except BudgetExceeded as e:
        print(f"unexpectedly blocked: {e}")
print(f"cost: {seen[-1].cost}")
for w in caught:
    print(f"warning: {type(w.message).__name__}")

## One line of truth

You do not have to find a rate card: name the MODEL the deployment serves and its rates are copied. An unknown base **raises** rather than leaving it quietly unpriced.

In [ ]:
by_base = prices.register_deployment(recipe.DEPLOYMENT, like=recipe.BASE_MODEL)
sorted(by_base)

## The same cap now binds

In [ ]:
reset()
blocked = False
try:
    with budget(usd=0.000_01, on_exceed="block"):
        recipe.ask(client)
except BudgetExceeded as e:
    blocked = True
    print(e)

## 5 · Prove it

In [ ]:
assert blocked, "the USD cap still did not bind AFTER registering the deployment"
reset()
with budget(usd=1.00, on_exceed="block"):
    recipe.ask(client)
assert seen[-1].cost and seen[-1].cost.amount > 0
print(f"OK — ${seen[-1].cost.amount}")